# Inline JavaScript Syntax Analysis

This notebook loads the relevant HTML files and checks their embedded JavaScript blocks for brace balance and syntax issues.

In [ ]:
from pathlib import Path
import re

root = Path(r"c:\Users\vondr\Documents\GitHub\ctrl-alt-del-inventory")

files = [root / 'public' / 'spirits.html', root / 'public' / 'findings.html']

script_pattern = re.compile(r'<script>(.*?)</script>', re.DOTALL | re.IGNORECASE)

for path in files:
    text = path.read_text(encoding='utf-8')
    scripts = script_pattern.findall(text)
    print(f"=== {path.name} ===")
    if not scripts:
        print('No inline <script> block found')
        continue
    for i, script in enumerate(scripts, 1):
        code = script.strip()
        brace_balance = 0
        paren_balance = 0
        bracket_balance = 0
        in_string = None
        escape = False
        for ch in code:
            if escape:
                escape = False
                continue
            if in_string:
                if ch == '\\':
                    escape = True
                elif ch == in_string:
                    in_string = None
                continue
            if ch in ('"', "'", '`'):
                in_string = ch
                continue
            if ch == '{':
                brace_balance += 1
            elif ch == '}':
                brace_balance -= 1
            elif ch == '(':
                paren_balance += 1
            elif ch == ')':
                paren_balance -= 1
            elif ch == '[':
                bracket_balance += 1
            elif ch == ']':
                bracket_balance -= 1
        print(f"Script {i}: braces={brace_balance}, parens={paren_balance}, brackets={bracket_balance}")
        if brace_balance != 0 or paren_balance != 0 or bracket_balance != 0:
            print('Possible imbalance detected')
        print()